# Louvain
        **graph › community_detection** · notebook `louvain`

        **Kapan dipakai:** Standar de facto deteksi komunitas: memaksimalkan modularitas, cepat dan skalabel.

        **Jebakan umum:** Batas resolusi: komunitas kecil bisa tergabung. Jumlah komunitas tidak bisa diatur langsung.

**Data demo:** (a) Karate Club Zachary (34 simpul; kebenaran = dua klub) dan (b) Cora komponen terbesar (kebenaran = 7 topik). NMI menilai kecocokan dengan kebenaran; **modularitas** menilai kualitas partisi tanpa label.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
MODEL_KEY = "louvain"; SMALL_ONLY = False      # metode yang hanya cocok graf kecil
OUT = out_dir("graph", "community_detection", MODEL_KEY); set_seed(42)
DATASETS = {"karate": ("data_latihan/graph/karate", "club", 2)}
if not SMALL_ONLY: DATASETS["cora (LCC)"] = ("data_latihan/graph/cora", "label", 7)

## 2. Metode
`communities(G, K)` mengembalikan daftar himpunan simpul (K = jumlah komunitas bila metode membutuhkannya).

In [ ]:
import numpy as np, networkx as nx
import numpy as np
import networkx as nx

def communities(G, K):
    return [set(c) for c in nx.community.louvain_communities(G, seed=42)]

## 3. Jalankan pada tiap graf dan nilai

In [ ]:
import time
from sklearn.metrics import normalized_mutual_info_score as nmi, adjusted_rand_score as ari
rows = []
for name, (folder, lab, K) in DATASETS.items():
    nodes = pd.read_csv(ROOT / folder / "nodes.csv"); edges = pd.read_csv(ROOT / folder / "edges.csv")
    G = nx.from_pandas_edgelist(edges, "source", "target"); G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()                            # komponen terhubung terbesar
    truth = nodes.set_index("id")[lab].astype(str).reindex(list(G.nodes)).to_numpy()
    t0 = time.time(); comms = communities(G, K); sec = time.time() - t0
    idx = {n: i for i, n in enumerate(G.nodes)}; pred = np.empty(len(G), int)
    for c, S in enumerate(comms):
        for n in S: pred[idx[n]] = c
    rows.append({"graf": name, "simpul": G.number_of_nodes(), "n_komunitas": len(comms), "NMI": nmi(truth, pred), "ARI": ari(truth, pred), "modularitas": nx.community.modularity(G, comms), "detik": sec})
res = pd.DataFrame(rows); display(res.round(3)); metrics = res.round(4).to_dict("records")

Modularitas tinggi tidak selalu berarti cocok dengan kebenaran: komunitas struktural (sisi padat) bisa berbeda dari label (topik/klub). Jumlah komunitas yang ditemukan juga penting: bandingkan dengan jumlah kelas sebenarnya.

In [ ]:
save_json({"results": metrics}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan Louvain dengan metode lain pada graf yang sama; atur parameter `resolution` Louvain untuk komunitas lebih kecil/besar.
- Untuk graf berlabel sebagian, gunakan `node_classification`; komunitas adalah cara tanpa label.